# Experiment 1 — ADE20K Linear Probe (T06 refactor, T07 clean-up)

**Protocol:** spatial no-regression check. DINOv2-base (timm wrapper, zero-shot for AnyUp3D), single images (T=1), 16² features → upsampler → 224² → 1×1 conv head → mIoU at 224² (images squashed to 224², masks nearest-resized).
**Conditions:** `bilinear`, `anyup2d`, `anyup3d`, through `Upsampler.forward_images` (images are never mixed).

Built on the shared modules; see `docs/plan/P1_experiments/T06_exp1_refactor.md`.

- Protocol: 3 seeds × 3 epochs, seeded shuffle (a seed gives every condition the same head init and data order), deterministic seeding (`anyup.utils.seed`).
- **`USE_T4 = True`**: test profile (small data, 1 seed). Its numbers are for checking the code, not for reporting.
- The T06 regression check against the old Exp 1 code (`LEGACY`, passed) was removed in T07; it is in git at commit `8d34f11`.

LR features are cached in fp16 on Drive and copied to local disk once per session. Head checkpoints on Drive allow resume: re-run the main cell after a disconnect. Results go to `<OUTPUT_ROOT>/<RUN_NAME>/`. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # T06 continues on the T02 branch
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
from pathlib import Path

USE_T4 = False   # ← True = T4 test profile (small data, numbers not for reporting); False = A100 real run

GPU = "T4" if USE_T4 else "A100"
GPU_PROFILES = {
    "A100": dict(BATCH_SIZE=16, Q_CHUNK=4096, N_TRAIN=None, N_VAL=None, NUM_EPOCHS=3, SEEDS=[42, 123, 456], NUM_WORKERS=4),
    "T4":   dict(BATCH_SIZE=4,  Q_CHUNK=2048, N_TRAIN=256,  N_VAL=64,   NUM_EPOCHS=1, SEEDS=[42],           NUM_WORKERS=2),
}
P = GPU_PROFILES[GPU]
BATCH_SIZE  = P["BATCH_SIZE"]    # A100: 16, T4: 4 — ↓ images per upsampler + head step; upsampled batch = B·768·IMG_SIZE²·4 B (2.5 GB at 16).
                                 #   16 is the protocol value (Q6). Dependent: Q_CHUNK, IMG_SIZE
Q_CHUNK     = P["Q_CHUNK"]       # A100: 4096, T4: 2048 — ↓ AnyUp3D query chunk → attention memory. Dependent: BATCH_SIZE, IMG_SIZE
N_TRAIN     = P["N_TRAIN"]       # A100: None (all 20,210), T4: 256 — ↓ train images → run time. Dependent: feature cache folder (CACHE_TAG)
N_VAL       = P["N_VAL"]         # A100: None (all 2,000), T4: 64 — ↓ val images → run time. Dependent: feature cache folder (CACHE_TAG)
NUM_EPOCHS  = P["NUM_EPOCHS"]    # A100: 3, T4: 1 — ↓ upsampler passes over the data. Dependent: LR_DECAY_STEP
SEEDS       = P["SEEDS"]         # A100: 42/123/456, T4: 42 — ↓ one full training per seed and condition; paired stats over seeds need ≥ 2
NUM_WORKERS = P["NUM_WORKERS"]   # A100: 4, T4: 2 — JPEG decode workers; no effect on results. Dependent: Colab CPU RAM

CKPT_ANYUP3D     = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # step6000 vs 6500 still open (plan.md)
ADE_ZIP          = "/content/drive/MyDrive/results/ADEChallengeData2016.zip"
ADE_DEST         = "/content"
OUTPUT_ROOT      = "/content/drive/MyDrive/results/T06/exp1"
CACHE_DRIVE_ROOT = "/content/drive/MyDrive/results/T06/feat_cache"   # kept between sessions (train ≈ 7.9 GB + 1 GB masks)
CACHE_LOCAL_ROOT = "/content/t06_feat_cache"                         # read during training (fast)
DEVICE           = "cuda"

CONDITIONS  = ["bilinear", "anyup2d", "anyup3d"]
BASELINES   = ["bilinear", "anyup2d"]   # paired_stats: anyup3d vs each of these
DINOV2_VARIANT = "base"

IMG_SIZE    = 224      # ↓ backbone input = guide = output = mask size; a multiple of 14. Upsampled batch ∝ IMG_SIZE².
                       #   Dependent: BATCH_SIZE, Q_CHUNK, feature grid (IMG_SIZE/14), feature cache folder (CACHE_TAG)
LR, WEIGHT_DECAY = 1e-3, 1e-4
LR_DECAY_STEP, LR_DECAY_GAMMA = 2, 0.1   # StepLR, in epochs. Depends on NUM_EPOCHS

RUN_VISUALS = False    # optional qualitative figure (Cell 7)
N_EXAMPLES  = 4        # ↓ rows of the figure; one upsampler call of this batch size per condition

CKPT_TAG = Path(CKPT_ANYUP3D).stem.rsplit("_", 1)[-1]
DATA_TAG = f"tr{N_TRAIN or 'all'}_va{N_VAL or 'all'}"
CACHE_TAG = f"dinov2_{DINOV2_VARIANT}_s{IMG_SIZE}_{DATA_TAG}"
RUN_NAME = f"{GPU}_{CKPT_TAG}_s{IMG_SIZE}_b{BATCH_SIZE}_q{Q_CHUNK}_{DATA_TAG}_e{NUM_EPOCHS}"
OUT_DIR  = Path(OUTPUT_ROOT) / RUN_NAME
HEAD_DIR = OUT_DIR / "head_ckpts"
CACHE_DRIVE = Path(CACHE_DRIVE_ROOT) / CACHE_TAG
CACHE_LOCAL = Path(CACHE_LOCAL_ROOT) / CACHE_TAG
print(f"GPU={GPU} RUN_NAME={RUN_NAME}\nCONDITIONS={CONDITIONS} SEEDS={SEEDS}")

In [ ]:
# ── Cell 3 · Imports, data, feature cache, upsamplers ──────────────────────
import random, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from anyup import ade20k, metrics
from anyup.ade20k import IGNORE_INDEX, NUM_CLASSES
from anyup.backbones import BackboneLoader, BackboneOutput, IMAGENET_MEAN, IMAGENET_STD
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed   # shared: seeds + deterministic cuDNN / algorithms (T07)

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if GPU not in GPU_NAME:
    print(f"WARNING: profile {GPU} but the runtime has {GPU_NAME}")

ROOT = ade20k.prepare_ade20k(ADE_DEST, drive_zip=ADE_ZIP)
PAIRS = {"train": ade20k.list_pairs(ROOT, "train")[:N_TRAIN], "val": ade20k.list_pairs(ROOT, "val")[:N_VAL]}
print({k: len(v) for k, v in PAIRS.items()})

bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=IMG_SIZE, guide_size=IMG_SIZE, variant=DINOV2_VARIANT)
BACKBONE_ID = bb.timm_name
for split, pairs in PAIRS.items():
    if ade20k.cache_complete(CACHE_DRIVE, split) and not ade20k.cache_complete(CACHE_LOCAL, split):
        t0 = time.time()
        ade20k.copy_cache(CACHE_DRIVE, CACHE_LOCAL, split)
        print(f"{split}: cache copied from Drive in {time.time() - t0:.0f} s")
    t0 = time.time()
    ade20k.build_feature_cache(bb, pairs, CACHE_LOCAL, split, batch_size=BATCH_SIZE,   # A100: 16, T4: 4 (Dependent: none, backbone only)
                               num_workers=NUM_WORKERS)                                # A100: 4, T4: 2
    print(f"{split}: cache ready in {time.time() - t0:.0f} s")
    if not ade20k.cache_complete(CACHE_DRIVE, split):
        ade20k.copy_cache(CACHE_LOCAL, CACHE_DRIVE, split)
del bb                     # backbone isn't needed again
torch.cuda.empty_cache()

train_ds = ade20k.CachedFeats(CACHE_LOCAL, "train", PAIRS["train"], IMG_SIZE, backbone_id=BACKBONE_ID)
val_ds   = ade20k.CachedFeats(CACHE_LOCAL, "val",   PAIRS["val"],   IMG_SIZE, backbone_id=BACKBONE_ID)
FEAT_DIM = train_ds[0][1].shape[0]
print(f"guide {tuple(train_ds[0][0].shape)}  feats {tuple(train_ds[0][1].shape)}  mask {tuple(train_ds[0][2].shape)}")

UPS = {}
if "bilinear" in CONDITIONS:
    UPS["bilinear"] = UpsamplerLoader.load("bilinear", device=DEVICE)
if "anyup2d" in CONDITIONS:
    UPS["anyup2d"] = UpsamplerLoader.load("anyup2d", device=DEVICE)
if "anyup3d" in CONDITIONS:
    UPS["anyup3d"] = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE,
                                          q_chunk_size=Q_CHUNK)   # A100: 4096, T4: 2048. Dependent: BATCH_SIZE, IMG_SIZE
print("upsamplers:", list(UPS))

In [ ]:
# ── Cell 4 · Head, train, evaluate, checkpoints ────────────────────────────
def make_head():
    return nn.Conv2d(FEAT_DIM, NUM_CLASSES, kernel_size=1).to(DEVICE)

def upsample(condition, guide, feats):
    # guide (B, 3, S, S), feats (B, C, h, w) on DEVICE → (B, C, S, S). forward_images: B unrelated images
    chunk = BackboneOutput(features=feats, guide=guide, meta={})
    return UPS[condition].forward_images(chunk, output_size=IMG_SIZE).features

def train_one_epoch(head, optimizer, condition, loader):
    head.train()
    losses = []
    for guide, feats, masks in loader:
        guide, feats = guide.to(DEVICE), feats.to(DEVICE)
        masks = masks.to(DEVICE) - 1                       # shift: 1–150 → 0–149; 0 → -1 (ignore)
        logits = head(upsample(condition, guide, feats))   # (B, 150, S, S)
        loss = F.cross_entropy(logits, masks, ignore_index=-1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

@torch.no_grad()
def evaluate(head, condition, loader, per_image):
    head.eval()
    conf_mat = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    scores = []
    for guide, feats, masks in loader:
        guide, feats, masks = guide.to(DEVICE), feats.to(DEVICE), masks.to(DEVICE)
        logits = head(upsample(condition, guide, feats))
        metrics.update_conf_mat(conf_mat, logits, masks, NUM_CLASSES, IGNORE_INDEX)
        if per_image:
            scores.extend(metrics.image_miou(logits, masks, NUM_CLASSES, IGNORE_INDEX))
    return metrics.compute_miou(conf_mat), conf_mat, scores

def head_path(condition, seed):
    return HEAD_DIR / f"{condition}_seed{seed}.pt"

def save_head(condition, seed, state):
    HEAD_DIR.mkdir(parents=True, exist_ok=True)
    path = head_path(condition, seed)
    tmp = path.with_suffix(".part")
    torch.save(state, tmp)
    os.replace(tmp, path)        # a disconnect during the save leaves the previous checkpoint intact

def run_seed(seed, condition):
    set_seed(seed)                                         # same seed → same head init for every condition
    head = make_head()
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEP, gamma=LR_DECAY_GAMMA)
    g = torch.Generator().manual_seed(seed)                # same seed → same data order for every condition
    history, start_epoch, scores = [], 0, None

    path = head_path(condition, seed)
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
        head.load_state_dict(ckpt["head"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        g.set_state(ckpt["gen_state"].cpu())
        history, start_epoch, scores = ckpt["history"], ckpt["epoch"], ckpt["per_image"]
        print(f"    resumed after epoch {start_epoch}: mIoU {ckpt['miou']:.2f}")
        if start_epoch >= NUM_EPOCHS:
            return ckpt["miou"], history, scores

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,       # A100: 16, T4: 4. Dependent: Q_CHUNK, IMG_SIZE
                              shuffle=True, generator=g,
                              num_workers=NUM_WORKERS,               # A100: 4, T4: 2
                              pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,         # A100: 16, T4: 4. Dependent: Q_CHUNK, IMG_SIZE
                              shuffle=False,
                              num_workers=NUM_WORKERS,               # A100: 4, T4: 2
                              pin_memory=True)

    for epoch in range(start_epoch + 1, NUM_EPOCHS + 1):             # A100: 3 epochs, T4: 1
        t0 = time.time()
        losses = train_one_epoch(head, optimizer, condition, train_loader)
        scheduler.step()
        t1 = time.time()
        last = epoch == NUM_EPOCHS
        miou, conf_mat, epoch_scores = evaluate(head, condition, val_loader, per_image=last)
        loss = float(np.mean(losses))
        history.append(dict(epoch=epoch, loss=loss, miou=miou, train_s=t1 - t0, val_s=time.time() - t1))
        print(f"    epoch {epoch:02d}/{NUM_EPOCHS}  loss={loss:.4f}  mIoU={miou:.2f}  "
              f"train {t1 - t0:.0f} s  val {time.time() - t1:.0f} s  "
              f"peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        scores = epoch_scores if last else None
        save_head(condition, seed, dict(head=head.state_dict(), optimizer=optimizer.state_dict(),
                                        scheduler=scheduler.state_dict(), gen_state=g.get_state(),
                                        epoch=epoch, miou=miou, history=history, per_image=scores))
    return miou, history, scores

In [ ]:
# ── Cell 5 · Main loop (re-run after a disconnect: finished work is skipped) ─
MIOU, HISTORY, PER_IMAGE = {}, {}, {}             # (condition, seed) → final mIoU, epoch history, per-image scores
t_start = time.time()
for condition in CONDITIONS:
    print(f"\n── {condition} " + "─" * 50)
    for seed in SEEDS:                            # A100: 42/123/456, T4: 42
        print(f"  seed {seed}")
        torch.cuda.reset_peak_memory_stats()
        MIOU[condition, seed], HISTORY[condition, seed], PER_IMAGE[condition, seed] = run_seed(seed, condition)
print(f"\nmain loop: {time.time() - t_start:.0f} s")

In [ ]:
# ── Cell 6 · Summary + save ────────────────────────────────────────────────
def fmt_p(p):
    return "n/a" if p is None else f"{p:.3g}"   # paired_stats gives None when a test can't be computed (e.g. all tied)

stems = [p[0].stem for p in PAIRS["val"]]
rows, summary = [], {"conditions": {}, "paired_seeds": {}, "paired_images": {}}
for condition in CONDITIONS:
    for seed in SEEDS:
        rows.extend(dict(image=s, condition=condition, seed=seed, image_miou=v)
                    for s, v in zip(stems, PER_IMAGE[condition, seed]))

print(f"{'condition':<10} {'seeds':<24} {'mean':>7} {'std':>6} {'image mIoU':>11}")
image_mean = {}   # condition → (n_images,) per-image mIoU averaged over seeds
for condition in CONDITIONS:
    seed_miou = [MIOU[condition, s] for s in SEEDS]
    image_mean[condition] = np.mean([PER_IMAGE[condition, s] for s in SEEDS], axis=0)
    mean, std = float(np.mean(seed_miou)), float(np.std(seed_miou, ddof=1)) if len(seed_miou) > 1 else 0.0
    summary["conditions"][condition] = dict(
        miou_per_seed=dict(zip(map(str, SEEDS), seed_miou)), miou_mean=mean, miou_std=std,
        history={str(s): HISTORY[condition, s] for s in SEEDS},
        image_miou_mean=float(np.nanmean(image_mean[condition])))
    print(f"{condition:<10} {' '.join(f'{v:.2f}' for v in seed_miou):<24} {mean:>7.2f} {std:>6.2f} "
          f"{np.nanmean(image_mean[condition]):>11.2f}")

keep = np.all([np.isfinite(image_mean[c]) for c in CONDITIONS], axis=0)
summary["n_images_scored"], summary["n_images_dropped"] = int(keep.sum()), int((~keep).sum())
print(f"\nper-image scores: {keep.sum()} images, {(~keep).sum()} dropped (no labelled pixel)")
for base in BASELINES:
    if "anyup3d" not in CONDITIONS or base not in CONDITIONS:
        continue
    r_img = metrics.paired_stats(image_mean["anyup3d"][keep], image_mean[base][keep])
    summary["paired_images"][f"anyup3d_vs_{base}"] = r_img
    print(f"anyup3d vs {base} per image: Δ {r_img['mean_delta']:+.3f}, better {r_img['n_improved']}/{r_img['n']}, "
          f"sign p2 {fmt_p(r_img['sign_p_two_sided'])}, Wilcoxon p2 {fmt_p(r_img['wilcoxon_p_two_sided'])}")
    if len(SEEDS) > 1:
        r_seed = metrics.paired_stats([MIOU["anyup3d", s] for s in SEEDS], [MIOU[base, s] for s in SEEDS])
        summary["paired_seeds"][f"anyup3d_vs_{base}"] = r_seed
        print(f"anyup3d vs {base} over seeds (n={r_seed['n']}, weak): Δ {r_seed['mean_delta']:+.3f}, "
              f"t-test p2 {fmt_p(r_seed['ttest_p_two_sided'])}")

up3d = UPS.get("anyup3d")
config = dict(gpu_profile=GPU, gpu_name=GPU_NAME, run_name=RUN_NAME, conditions=CONDITIONS,
              seeds=SEEDS, batch_size=BATCH_SIZE, q_chunk=Q_CHUNK, n_train=len(train_ds), n_val=len(val_ds),
              num_epochs=NUM_EPOCHS, img_size=IMG_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
              lr_decay_step=LR_DECAY_STEP, lr_decay_gamma=LR_DECAY_GAMMA, backbone=BACKBONE_ID,
              anyup3d_meta=up3d._extra_meta() if up3d else None,
              anyup2d_meta=UPS["anyup2d"]._extra_meta() if "anyup2d" in UPS else None,
              cache=str(CACHE_DRIVE), shuffle=True, repo_branch=REPO_BRANCH,
              ade20k_stems_sha1={s: ade20k.cache_meta(CACHE_LOCAL, s)["stems_sha1"] for s in PAIRS})
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp1")
print(f"\nsaved {csv_path}\n      {json_path}")

In [ ]:
# ── Cell 7 · Optional qualitative figure (trained heads, first seed) ───────
if RUN_VISUALS:
    rng = np.random.default_rng(0)
    palette = rng.integers(0, 255, size=(NUM_CLASSES + 1, 3), dtype=np.uint8)
    palette[0] = 0
    mean = torch.tensor(IMAGENET_MEAN)[:, None, None]
    std  = torch.tensor(IMAGENET_STD)[:, None, None]

    guide, feats, masks = next(iter(DataLoader(val_ds, batch_size=N_EXAMPLES, shuffle=False)))   # ↓ N_EXAMPLES images in one upsampler call
    preds = {}
    for condition in CONDITIONS:
        head = make_head()
        head.load_state_dict(torch.load(head_path(condition, SEEDS[0]), map_location=DEVICE, weights_only=True)["head"])
        head.eval()
        with torch.no_grad():
            logits = head(upsample(condition, guide.to(DEVICE), feats.to(DEVICE)))
        preds[condition] = logits.argmax(dim=1).cpu().numpy() + 1                # back to ids 1..150

    ncols = 2 + len(CONDITIONS)
    fig, axes = plt.subplots(N_EXAMPLES, ncols, figsize=(4 * ncols, 4 * N_EXAMPLES), squeeze=False)
    for col, title in enumerate(["Image", "GT"] + CONDITIONS):
        axes[0][col].set_title(title, fontsize=13, fontweight="bold")
    for row in range(N_EXAMPLES):
        axes[row][0].imshow((guide[row] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[row][1].imshow(palette[masks[row].numpy()])
        for col, condition in enumerate(CONDITIONS, start=2):
            axes[row][col].imshow(palette[preds[condition][row]])
        for ax in axes[row]:
            ax.axis("off")
    fig.tight_layout(pad=1.0)
    fig_path = OUT_DIR / "exp1_qualitative.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved {fig_path}")